# Packages and options

In [1]:
import pandas as pd
import numpy as np
import gdxtools as gt
import gams
import os
import sys

In [2]:
hasattr(gt, 'add_set')

True

In [3]:
fn_gdx = "./test.gdx"

# GAMS Workspace

Using the standard gams API, create workspace with working director in the current directory:

In [4]:
dir_gms = os.getcwd()
ws = gams.GamsWorkspace(dir_gms)

# Reading a gdx file

Add gdx database from file:

In [4]:
gdx = ws.add_database_from_gdx(fn_gdx)

## Reading sets

Sets are returned as python lists

In [5]:
gt.get_symbol_values(gdx, "r")

['i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']

## Reading Scalar

Scalars are returned as scalar values:

In [6]:
gt.get_symbol_values(gdx, "test_scalar")

1.0

## Reading Parameters

All parameters are returned as pandas series possibly with mulit-index for the respective set dimension

In [7]:
df_one = gt.get_symbol_values(gdx, "one_dim", col_names=["items"])
df_one.head()

items
i1    17.174713
i2    84.326671
i3    55.037536
i4    30.113790
i5    29.221212
Name: Value, dtype: float64

In [8]:
df_two = gt.get_symbol_values(gdx, "two_dim", col_names=["item", "region"])
df_two.head()

item  region
i1    i1         5.140711
      i2         0.600837
      i3        40.122768
      i4        51.988119
      i5        62.887726
Name: Value, dtype: float64

# Write to gdx

Create a new gdx file (within the same workspace to avoid confusions):

In [9]:
gdx2 = ws.add_database()

## Adding a set 

Sets are passed as list. We can specify whether we want to overwrite existing values. By default an error is thrown if the items already exists.

In [10]:
l = ["i%d" % i for i in range(1,5)]
gt.add_set(l, "i", gdx2, text="these are some test items", append=False, overwrite=True)
gt.get_symbol_values(gdx2, "i")

['i1', 'i2', 'i3', 'i4']

Append some additional values (note that you have to set overwrite true for this... should be changed...):

In [13]:
gt.add_set(["i%d" % i for i in range(5,10)], 
           "i", gdx2, text="these are some test items", append=True, overwrite=True)
gt.get_symbol_values(gdx2, "i")

['i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9']

## Adding parameters 

First add a scalar value. Again, by defaul an error is thrown if the item already exists. 

In [14]:
gt.add_parameter(5.2, "scalar_test", gdx2, text="test of scalar values", append=False, overwrite=True)
gt.get_symbol_values(gdx2, "scalar_test")

5.2

Parameters are passed as dictionaries. Note that domain checkking is not provided.

In [15]:
one_test = {"i%d" % i: np.random.uniform(0,100) for i in range(1,5)}
gt.add_parameter(one_test, "one_par", gdx2, text="test of a one-dimensional parameter", append=False, overwrite=True)
gt.get_symbol_values(gdx2, "one_par")

dim1
i1    83.997737
i2    58.851909
i3    99.163476
i4    26.245279
Name: Value, dtype: float64

And we can append to paramters (but will possibly overwrite, if values already exist)

In [16]:
gt.add_parameter({"i%d" % i: np.random.uniform(0,100) for i in range(5,15)}, 
                 "one_par", gdx2, text="test of a one-dimensional parameter", append=True, overwrite=True)
gt.get_symbol_values(gdx2, "one_par")

dim1
i1     83.997737
i2     58.851909
i3     99.163476
i4     26.245279
i5     41.124298
i6     19.731398
i7     85.484380
i8     81.608289
i9     48.247914
i10     1.780452
i11    25.253234
i12    37.289890
i13    17.514183
i14    68.614849
Name: Value, dtype: float64

## Save gdx

GDX is saved using the standard gams API

In [17]:
gdx2.export("./write_text.gdx")